# HRV-COVID19 Data Exploration

This notebook explores the Welltory HRV-COVID19 dataset. The analysis focuses on daily wearable measurements, environmental conditions, and participant context, while using blood-pressure events and surveys as supporting tables. The goal is to understand data quality and relationships without treating observational associations as causal evidence.

## Data context, sampling, and provenance

The source repository is [Welltory/hrv-covid19](https://github.com/Welltory/hrv-covid19). It contains de-identified personal health, activity, sleep, weather, blood-pressure, and survey records collected from users during the COVID-19 period. The sample is observational and appears to be a convenience sample of users with different devices and recording habits, so missingness is informative and the findings should not be generalized to a population.

The CSV files were downloaded from the repository's `data/` directory into the local `data/hrv_covid19/` folder. That folder is listed in `.gitignore`, so the downloaded data is intentionally not committed. The daily analysis table uses `user_code` and `day` as its join keys; event tables retain their own event timestamps.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path('data/hrv_covid19')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/hrv_covid19')

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 40)
print(f'Data directory: {DATA_DIR.resolve()}')

In [ ]:
# Load each table separately because the tables have different grains.
tables = {name: pd.read_csv(DATA_DIR / f'{name}.csv') for name in [
    'participants', 'wearables', 'sleep', 'weather',
    'blood_pressure', 'surveys', 'scales_description'
]}
for name, table in tables.items():
    print(f'{name:20s} {table.shape[0]:>7,} rows x {table.shape[1]:>2} columns')

participants = tables['participants'].copy()
wearables = tables['wearables'].copy()
sleep = tables['sleep'].copy()
weather = tables['weather'].copy()
blood_pressure = tables['blood_pressure'].copy()
surveys = tables['surveys'].copy()

## Structure and descriptive statistics

I first inspect shape, data types, date ranges, and descriptive statistics. This prevents applying a daily aggregation or numeric operation to an event table by mistake. Numeric columns are converted explicitly because the CSV uses empty strings for many missing measurements.

In [ ]:
# Normalize dates and numeric fields before profiling.
for frame in [wearables, sleep, weather]:
    frame['day'] = pd.to_datetime(frame['day'], errors='coerce')
blood_pressure['measurement_datetime'] = pd.to_datetime(
    blood_pressure['measurement_datetime'], errors='coerce'
)
surveys['created_at'] = pd.to_datetime(surveys['created_at'], errors='coerce')

for frame in [participants, wearables, sleep, weather, blood_pressure, surveys]:
    for column in frame.select_dtypes(include='object').columns:
        if column not in ['user_code', 'gender', 'age_range', 'city', 'country', 'scale', 'text']:
            converted = pd.to_numeric(frame[column], errors='coerce')
            if converted.notna().sum() > 0:
                frame[column] = converted

participants.describe(include='all').T.head(12)

In [ ]:
print('Participant coverage:')
print(participants[['gender', 'age_range', 'country']].nunique())
print(f"Participant rows: {len(participants):,}; unique users: {participants['user_code'].nunique():,}")
print(f"Wearable date range: {wearables['day'].min().date()} to {wearables['day'].max().date()}")
print(f"Wearable users: {wearables['user_code'].nunique():,}")
print('Age range counts:')
display(participants['age_range'].value_counts(dropna=False).sort_index())

## Data quality assessment

Completeness is reported as non-null percentage because device-derived fields can be absent when a user did not wear or sync a device. I check duplicate keys separately for each table: `user_code` + `day` should be unique for daily tables, while repeated blood-pressure and survey records are valid event observations. Accuracy cannot be fully verified without the original device metadata, but ranges and consistency checks can identify implausible values.

In [ ]:
def quality_report(frame):
    return pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing_count': frame.isna().sum(),
        'missing_pct': (frame.isna().mean() * 100).round(1),
        'unique_count': frame.nunique(dropna=True),
    }).sort_values('missing_pct', ascending=False)

print('Wearables completeness:')
display(quality_report(wearables))

for name, frame in {'wearables': wearables, 'sleep': sleep, 'weather': weather}.items():
    duplicate_count = frame.duplicated(['user_code', 'day']).sum()
    print(f'{name}: duplicate user-day rows = {duplicate_count:,}')

print('Range checks:')
print('Negative steps:', (wearables['steps_count'] < 0).sum())
print('SpO2 outside 0-100:', ((wearables['average_spo2_value'] < 0) | (wearables['average_spo2_value'] > 100)).sum())
print('Blood pressure with systolic <= diastolic:', (blood_pressure['systolic'] <= blood_pressure['diastolic']).sum())

## Cleaning and transformation choices

I remove exact duplicate daily keys only after checking their count, because duplicated rows would overweight a user-day. I do not fill all missing values globally: a missing step count may mean no device data, not zero steps. For exploratory summaries, I use median imputation within the cleaned daily table because medians are less sensitive to skew and extreme activity days. I keep a missingness indicator for steps so the imputation is not mistaken for an observed value.

The new features are `bmi` from height and weight and `sleep_hours` from sleep duration. BMI is used only as a descriptive feature because height/weight may be self-reported, and sleep duration is converted from seconds to hours for interpretability.

In [ ]:
# Build one daily analysis table from tables that share user_code and day.
daily = wearables.drop_duplicates(['user_code', 'day']).merge(
    weather.drop_duplicates(['user_code', 'day']),
    on=['user_code', 'day'], how='left', suffixes=('', '_weather')
)
sleep_daily = sleep.drop_duplicates(['user_code', 'day'])[['user_code', 'day', 'sleep_duration']].copy()
sleep_daily['sleep_hours'] = sleep_daily['sleep_duration'] / 3600
daily = daily.merge(sleep_daily[['user_code', 'day', 'sleep_hours']], on=['user_code', 'day'], how='left')

participant_features = participants[['user_code', 'gender', 'age_range', 'country', 'height', 'weight']].copy()
participant_features['height_m'] = participant_features['height'] / 100
participant_features['bmi'] = participant_features['weight'] / participant_features['height_m'].pow(2)
daily = daily.merge(participant_features[['user_code', 'gender', 'age_range', 'country', 'bmi']], on='user_code', how='left')

daily['steps_was_missing'] = daily['steps_count'].isna().astype(int)
numeric_for_eda = ['steps_count', 'resting_pulse', 'pulse_average', 'average_spo2_value', 'humidity', 'sleep_hours', 'bmi']
for column in numeric_for_eda:
    daily[column] = daily[column].fillna(daily[column].median())

print(f'Cleaned daily table: {daily.shape[0]:,} rows x {daily.shape[1]} columns')
display(daily.head())

## Outlier strategy

Health and activity variables are often skewed, so I identify outliers with the 1.5 x IQR rule rather than assuming a normal distribution. I retain the original values for transparency, create a count of outlier flags, and use clipped values only for a robust visualization. This limits the influence of extreme values without deleting potentially meaningful health observations.

In [ ]:
outlier_columns = ['steps_count', 'resting_pulse', 'pulse_average', 'sleep_hours', 'bmi']
outlier_flags = pd.DataFrame(index=daily.index)
for column in outlier_columns:
    q1, q3 = daily[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_flags[column] = ~daily[column].between(lower, upper)
    daily[f'{column}_clipped'] = daily[column].clip(lower, upper)
daily['outlier_count'] = outlier_flags.sum(axis=1)
print('Outlier counts by variable:')
display(outlier_flags.sum().sort_values(ascending=False).to_frame('outlier_rows'))

## Visual exploration

The first plot shows whether activity is concentrated in a small number of user-days. The second compares pulse and activity, while the third checks whether sleep duration differs visibly by age range. Scatterplots show association only; they do not establish that one variable causes another.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=daily, x='steps_count_clipped', bins=30, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Daily steps (IQR-clipped)')
axes[0].set_xlabel('Steps')
sns.scatterplot(data=daily.sample(min(1500, len(daily)), random_state=42), x='steps_count_clipped', y='pulse_average', alpha=0.35, ax=axes[1], color='#e76f51')
axes[1].set_title('Steps and average pulse')
sns.boxplot(data=daily, x='age_range', y='sleep_hours', ax=axes[2], color='#457b9d')
axes[2].set_title('Sleep duration by age range')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()

In [ ]:
# Correlation is computed on numeric daily fields to summarize linear relationships.
corr_columns = ['steps_count', 'resting_pulse', 'pulse_average', 'average_spo2_value', 'humidity', 'sleep_hours', 'bmi']
plt.figure(figsize=(9, 6))
sns.heatmap(daily[corr_columns].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Daily numeric correlation matrix')
plt.tight_layout()
daily[corr_columns].corr()['steps_count'].sort_values(ascending=False)

## Supporting tables: surveys and blood pressure

Surveys contain categorical scale codes and free-text descriptions, so I summarize response frequencies instead of treating the codes as continuous measurements. Blood-pressure records are event-level and may contain multiple readings per user-day; I summarize systolic and diastolic distributions without duplicating them into the daily wearable table.

In [ ]:
print('Most common survey scales:')
display(surveys['scale'].value_counts().head(10).to_frame('responses'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=blood_pressure, x='systolic', bins=25, ax=axes[0], color='#f4a261'])
axes[0].set_title('Systolic blood pressure events')
sns.countplot(data=surveys, y='scale', order=surveys['scale'].value_counts().head(10).index, ax=axes[1], color='#264653')
axes[1].set_title('Top survey scales')
plt.tight_layout()

## Findings and limitations

* The tables have different grains, so joining only on `user_code` and `day` avoids multiplying event records.
* Missingness is substantial in several device fields and is likely related to device availability or user behavior; median imputation is used only for exploratory summaries and is flagged for steps.
* The IQR rule identifies extreme values but does not prove that they are errors. Clipping is used for plots while original measurements remain available.
* The engineered BMI and sleep-hours features improve interpretability, but BMI may reflect self-reported measurements and sleep records may be incomplete.
* Data accuracy and integrity cannot be fully established from CSV files alone. The source repository provides provenance, but device calibration, collection protocols, and sampling bias remain limitations.
* Associations in this observational, de-identified convenience sample should not be interpreted as evidence that activity, sleep, weather, or pulse causes COVID-19 symptoms.